**Archived legacy method.** Set `YEAR` and, only when intentionally writing disposable outputs, set `SAVE = True`. Outputs go to `notebooks/outputs/<YEAR>/`; this notebook does not publish canonical pipeline data.

This notebook preserves the exploratory historical analysis of Michelin Guide changes. It does not establish a durable restaurant identity system and it does not mutate accepted annual products. Identity matching is shown separately from classification changes so promotions and demotions do not masquerade as new or removed restaurants.


In [133]:
# Legacy notebook controls: edit only these values.
YEAR = 2026
SAVE = False

PREVIOUS_YEAR = YEAR - 1

In [134]:
from pathlib import Path

REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent

import sys

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

RAW_ROOT = REPO_ROOT / "data" / "raw"
PARTITION_ROOT = REPO_ROOT / "data" / "partitions"
PRODUCT_ROOT = REPO_ROOT / "data" / "products"
OUTPUT_ROOT = REPO_ROOT / "notebooks" / "outputs" / str(YEAR)
GEODATA_OUTPUT_ROOT = OUTPUT_ROOT / "geodata"

DEPARTMENTS_PATH = RAW_ROOT / "demographics" / "departments.csv"
DEPARTMENT_STATS_PATH = RAW_ROOT / "demographics" / "departmental_stats_2023.csv"
ARRONDISSEMENT_STATS_PATH = RAW_ROOT / "demographics" / "arrondissement_stats_2023.csv"
DEPARTMENTS_GEOMETRY_PATH = RAW_ROOT / "geodata" / "departments.geojson"
REGIONS_GEOMETRY_PATH = RAW_ROOT / "geodata" / "regions.geojson"
ARRONDISSEMENTS_GEOMETRY_PATH = RAW_ROOT / "geodata" / "arrondissements-avec-outre-mer.geojson"
PARIS_GEOMETRY_PATH = RAW_ROOT / "geodata" / "paris_arrondissements.geojson"
MONACO_GEOMETRY_PATH = RAW_ROOT / "geodata" / "monaco.geojson"


In [135]:
def annual_restaurant_product(year):
    candidates = [
        PRODUCT_ROOT / "france" / str(year) / "all_restaurants(arrondissements).csv",
        PRODUCT_ROOT / "france" / f"all_restaurants(arrondissements)_{year % 100:02d}.csv",
        REPO_ROOT / "Years" / str(year) / "data" / "France" / "all_restaurants(arrondissements).csv",
    ]
    for candidate in candidates:
        if candidate.is_file():
            return candidate
    raise FileNotFoundError(f"No accepted annual restaurant product for {year}: {candidates}")


def with_comparison_metadata(frame):
    result = frame.copy()
    result.insert(0, "previous_year", PREVIOUS_YEAR)
    result.insert(1, "current_year", YEAR)
    return result


def save_csv(frame, stem):
    target = OUTPUT_ROOT / f"{stem}_{PREVIOUS_YEAR}_{YEAR}.csv"
    if not SAVE:
        print(f"SAVE=False: not writing {target}")
        return
    target.parent.mkdir(parents=True, exist_ok=True)
    with_comparison_metadata(frame).to_csv(target, index=False)
    print(f"Wrote {target}")


def save_geojson(frame, stem):
    target = GEODATA_OUTPUT_ROOT / f"{stem}_{PREVIOUS_YEAR}_{YEAR}.geojson"
    if not SAVE:
        print(f"SAVE=False: not writing {target}")
        return
    target.parent.mkdir(parents=True, exist_ok=True)
    frame.to_file(target, driver="GeoJSON")
    print(f"Wrote {target}")


In [136]:
import math
import re
import unicodedata
from difflib import SequenceMatcher

import pandas as pd
from IPython.display import Markdown, display

pd.set_option("display.max_colwidth", 120)

display(Markdown(f"# Comparing the {PREVIOUS_YEAR} and {YEAR} Michelin Guide to France"))


# Comparing the 2025 and 2026 Michelin Guide to France

## Load accepted annual products

The accepted annual product files are read once and then copied into working dataframes. The source snapshots remain unchanged throughout the analysis.


In [137]:
previous_product_path = annual_restaurant_product(PREVIOUS_YEAR)
current_product_path = annual_restaurant_product(YEAR)

france_previous_source = pd.read_csv(previous_product_path)
france_current_source = pd.read_csv(current_product_path)

france_previous = france_previous_source.copy(deep=True)
france_current = france_current_source.copy(deep=True)

print(f"Previous guide ({PREVIOUS_YEAR}): {previous_product_path} rows={len(france_previous)}")
print(f"Current guide ({YEAR}): {current_product_path} rows={len(france_current)}")


Previous guide (2025): /Users/Ian/Documents/Study/Stage3/Programming/Github/Michelin_Rated_Restaurants/data/products/france/2025/all_restaurants(arrondissements).csv rows=2985
Current guide (2026): /Users/Ian/Documents/Study/Stage3/Programming/Github/Michelin_Rated_Restaurants/data/products/france/2026/all_restaurants(arrondissements).csv rows=3055


In [138]:
france_previous.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2985 entries, 0 to 2984
Data columns (total 16 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   name            2985 non-null   object 
 1   address         2985 non-null   object 
 2   location        2985 non-null   object 
 3   arrondissement  2985 non-null   object 
 4   department_num  2985 non-null   object 
 5   department      2985 non-null   object 
 6   capital         2985 non-null   object 
 7   region          2985 non-null   object 
 8   price           2985 non-null   object 
 9   cuisine         2985 non-null   object 
 10  url             2831 non-null   object 
 11  award           2985 non-null   object 
 12  stars           2985 non-null   float64
 13  greenstar       2985 non-null   int64  
 14  longitude       2985 non-null   float64
 15  latitude        2985 non-null   float64
dtypes: float64(3), int64(1), object(12)
memory usage: 373.3+ KB


In [139]:
france_current.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3055 entries, 0 to 3054
Data columns (total 16 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   name            3055 non-null   object 
 1   address         3055 non-null   object 
 2   location        3055 non-null   object 
 3   arrondissement  3055 non-null   object 
 4   department_num  3055 non-null   object 
 5   department      3055 non-null   object 
 6   capital         3055 non-null   object 
 7   region          3055 non-null   object 
 8   price           3055 non-null   object 
 9   cuisine         3055 non-null   object 
 10  url             2894 non-null   object 
 11  award           3055 non-null   object 
 12  stars           3055 non-null   float64
 13  greenstar       3055 non-null   int64  
 14  longitude       3055 non-null   float64
 15  latitude        3055 non-null   float64
dtypes: float64(3), int64(1), object(12)
memory usage: 382.0+ KB


In [140]:
non_selected_current = france_current[france_current["stars"] != 0.25]
print(f"{YEAR} entries excluding Selected Restaurants: {len(non_selected_current)}")


2026 entries excluding Selected Restaurants: 1088


## Diagnostic: why the old tuple method was misleading

The original notebook compared full row tuples, and later `(name, location, stars)` tuples, to decide whether a restaurant was present in both years. Including `stars` in an identity key mixes two separate questions:

1. do two rows describe the same restaurant?
2. did that restaurant's Guide classification change?

A promotion or demotion changes the `stars` value, so a restaurant can be counted as both new and removed before the notebook even reaches change detection.


In [141]:
previous_starred = france_previous[france_previous["stars"] >= 1][["name", "address", "location", "url", "stars"]].copy()
current_starred = france_current[france_current["stars"] >= 1][["name", "address", "location", "url", "stars"]].copy()

legacy_previous_star_tuples = set(map(tuple, previous_starred[["name", "location", "stars"]].to_numpy()))
legacy_current_star_tuples = set(map(tuple, current_starred[["name", "location", "stars"]].to_numpy()))
legacy_tuple_new = legacy_current_star_tuples - legacy_previous_star_tuples
legacy_tuple_removed = legacy_previous_star_tuples - legacy_current_star_tuples

identity_without_stars_previous = set(map(tuple, previous_starred[["name", "location"]].to_numpy()))
identity_without_stars_current = set(map(tuple, current_starred[["name", "location"]].to_numpy()))

legacy_tuple_diagnostic = pd.DataFrame([
    {"check": "legacy tuples including stars: current not previous", "count": len(legacy_tuple_new)},
    {"check": "legacy tuples including stars: previous not current", "count": len(legacy_tuple_removed)},
    {"check": "name+location current not previous", "count": len(identity_without_stars_current - identity_without_stars_previous)},
    {"check": "name+location previous not current", "count": len(identity_without_stars_previous - identity_without_stars_current)},
])
legacy_tuple_diagnostic


,check,count
0,legacy tuples including stars: current not previous,92
1,legacy tuples including stars: previous not current,80
2,name+location current not previous,81
3,name+location previous not current,69


## Prepare matching fields

The matching fields below are derived columns for analysis only. They are not written back to the accepted annual products.


In [142]:
def normalize_text(value):
    if pd.isna(value):
        return ""
    text = unicodedata.normalize("NFKD", str(value).casefold())
    text = "".join(character for character in text if not unicodedata.combining(character))
    return re.sub(r"[^a-z0-9]+", " ", text).strip()


def normalize_url(value):
    if pd.isna(value) or not str(value).strip():
        return ""
    text = str(value).strip().casefold().split("?", 1)[0].split("#", 1)[0].rstrip("/")
    return text.replace("https://www.", "https://").replace("http://www.", "http://")


def postal_code(value):
    match = re.search(r"\b\d{5}\b", "" if pd.isna(value) else str(value))
    return match.group(0) if match else ""


def add_matching_fields(frame, year_label):
    working = frame.copy(deep=True)
    working["source_row"] = range(len(working))
    working["guide_year"] = year_label
    working["normalized_name"] = working["name"].map(normalize_text)
    working["normalized_location"] = working["location"].map(normalize_text)
    working["normalized_address"] = working.get("address", pd.Series("", index=working.index)).map(normalize_text)
    working["normalized_url"] = working.get("url", pd.Series("", index=working.index)).map(normalize_url)
    working["postal_code"] = working["location"].map(postal_code)
    return working


previous_work = add_matching_fields(france_previous, PREVIOUS_YEAR)
current_work = add_matching_fields(france_current, YEAR)


## Deterministic identity matching

These are conservative one-to-one matches used for analysis. `stars` and `award` are deliberately excluded from identity keys. The rules still have limitations: URL changes, relocation, duplicate names, or renamed restaurants can remain unmatched and require review.


In [143]:
def unique_key_matches(previous, current, previous_available, current_available, columns, method):
    if not previous_available or not current_available:
        return []

    previous_keys = (
        previous.loc[sorted(previous_available), ["source_row", *columns]]
        .fillna("")
        .astype(str)
    )
    current_keys = (
        current.loc[sorted(current_available), ["source_row", *columns]]
        .fillna("")
        .astype(str)
    )

    previous_keys = previous_keys[previous_keys[list(columns)].ne("").all(axis=1)].copy()
    current_keys = current_keys[current_keys[list(columns)].ne("").all(axis=1)].copy()
    previous_keys["key"] = previous_keys[list(columns)].agg("|".join, axis=1)
    current_keys["key"] = current_keys[list(columns)].agg("|".join, axis=1)

    previous_counts = previous_keys["key"].value_counts()
    current_counts = current_keys["key"].value_counts()
    previous_unique = previous_keys[previous_keys["key"].map(previous_counts).eq(1)].set_index("key")
    current_unique = current_keys[current_keys["key"].map(current_counts).eq(1)].set_index("key")

    matches = []
    for key in sorted(set(previous_unique.index) & set(current_unique.index)):
        matches.append({
            "previous_row": int(previous_unique.loc[key, "source_row"]),
            "current_row": int(current_unique.loc[key, "source_row"]),
            "matching_method": method,
            "match_evidence": key,
        })
    return matches


previous_available = set(previous_work["source_row"])
current_available = set(current_work["source_row"])
match_records = []

matching_rules = [
    (("normalized_url", "postal_code"), "url_postal"),
    (("normalized_url", "normalized_location"), "url_location"),
    (("normalized_url",), "url"),
    (("normalized_name", "normalized_location", "normalized_url"), "name_location_url"),
    (("normalized_name", "postal_code"), "name_postal"),
    (("normalized_name", "normalized_location"), "name_location"),
]

for columns, method in matching_rules:
    for match in unique_key_matches(previous_work, current_work, previous_available, current_available, columns, method):
        previous_row = match["previous_row"]
        current_row = match["current_row"]
        if previous_row in previous_available and current_row in current_available:
            match_records.append(match)
            previous_available.remove(previous_row)
            current_available.remove(current_row)

matches = pd.DataFrame(match_records)
print(f"Deterministic matches: {len(matches)}")
print(f"Unmatched previous rows: {len(previous_available)}")
print(f"Unmatched current rows: {len(current_available)}")
matches["matching_method"].value_counts().rename_axis("matching_method").reset_index(name="matches")


Deterministic matches: 2685
Unmatched previous rows: 300
Unmatched current rows: 370


,matching_method,matches
0,url_postal,1664
1,name_postal,956
2,name_location_url,59
3,url,6


## Classification changes after identity matching

Only after a row pair is matched do we compare `stars` and `award`. A restaurant absent from the next dataset is described as `removed_from_guide`, not closed.


In [144]:
def classify_star_change(previous_stars, current_stars):
    if current_stars > previous_stars:
        return "promoted"
    if current_stars < previous_stars:
        return "demoted"
    return "unchanged"


def comparison_records(matches, previous, current):
    records = []
    for row in matches.itertuples(index=False):
        old = previous.loc[previous["source_row"].eq(row.previous_row)].iloc[0]
        new = current.loc[current["source_row"].eq(row.current_row)].iloc[0]
        status = classify_star_change(old["stars"], new["stars"])
        records.append({
            "previous_row": row.previous_row,
            "current_row": row.current_row,
            "previous_name": old["name"],
            "current_name": new["name"],
            "previous_location": old["location"],
            "current_location": new["location"],
            "previous_stars": old["stars"],
            "current_stars": new["stars"],
            "previous_award": old.get("award"),
            "current_award": new.get("award"),
            "status": status,
            "matching_method": row.matching_method,
            "match_evidence": row.match_evidence,
        })
    return pd.DataFrame(records)


matched_comparison = comparison_records(matches, previous_work, current_work)
star_changes = matched_comparison[matched_comparison["status"].ne("unchanged")].copy()
star_changes.sort_values(["current_stars", "previous_stars", "current_name"], ascending=[False, False, True], inplace=True)
star_changes


,previous_row,current_row,previous_name,current_name,previous_location,current_location,previous_stars,current_stars,previous_award,current_award,status,matching_method,match_evidence
2452,65,1549,Les Morainières,Les Morainières,"Jongieux, 73170","Jongieux, 73170",2.0,3.00,2 Stars,3 Stars,promoted,name_postal,les morainieres|73170
58,0,1619,L'Ambroisie,L'Ambroisie,"Paris, 75004","Paris, 75004",3.0,2.00,3 Stars,2 Stars,demoted,url_postal,https://ambroisie-paris.com|75004
1240,143,1579,Alliance,Alliance,"Paris, 75005","Paris, 75005",1.0,2.00,1 Star,2 Stars,promoted,url_postal,https://restaurant-alliance.fr|75005
1766,129,1599,Arbane,Arbane,"Reims, 51100","Reims, 51100",1.0,2.00,1 Star,2 Stars,promoted,name_postal,arbane|51100
1682,114,1598,Bulle d'Osier,Bulle d'Osier,"Langres, 52200","Langres, 52200",1.0,2.00,1 Star,2 Stars,promoted,name_location_url,bulle d osier|langres 52200|https://closvauban.com
...,...,...,...,...,...,...,...,...,...,...,...,...,...
406,867,602,Racines,Racines,"Bourg-en-Bresse, 01000","Saint-Denis-lès-Bourg, 01000",0.5,0.25,Bib Gourmand,Selected Restaurants,demoted,url_postal,https://domainedulac-racines.fr|01000
2593,962,836,Restaurant Anne & Matthieu Omont - Hôtel de France,Restaurant Anne & Matthieu Omont - Hôtel de France,"Montmarault, 03390","Montmarault, 03390",0.5,0.25,Bib Gourmand,Selected Restaurants,demoted,name_postal,restaurant anne matthieu omont hotel de france|03390
1553,727,954,Sauf Imprévu,Sauf Imprévu,"Lyon, 69006","Lyon, 69006",0.5,0.25,Bib Gourmand,Selected Restaurants,demoted,url_postal,https://sauf-imprevu.fr|69006
2667,661,2634,Turbulent,Turbulent,"Trouville-sur-Mer, 14360","Trouville-sur-Mer, 14360",0.5,0.25,Bib Gourmand,Selected Restaurants,demoted,name_postal,turbulent|14360


In [145]:
potential_new_starred_entries = current_work[
    current_work["source_row"].isin(current_available) & current_work["stars"].ge(1)
].copy()
potential_removed_starred_entries = previous_work[
    previous_work["source_row"].isin(previous_available) & previous_work["stars"].ge(1)
].copy()

print(f"Potential starred entries new to the {YEAR} dataset: {len(potential_new_starred_entries)}")
print(f"Potential starred entries removed from the {YEAR} dataset: {len(potential_removed_starred_entries)}")


Potential starred entries new to the 2026 dataset: 48
Potential starred entries removed from the 2026 dataset: 41


In [146]:
potential_new_starred_entries[["source_row", "name", "location", "url", "award", "stars"]].sort_values(["stars", "name"], ascending=[False, True])


,source_row,name,location,url,award,stars
1602,1602,L'Abysse Paris,"Paris, 75008",https://www.yannick-alleno.com/les-etablissements-du-groupe/abysse-paris,2 Stars,2.0
1560,1560,La Table de Christophe Dufossé,"Busnes, 62350",https://www.ledomainedebeaulieu.fr,2 Stars,2.0
1781,1781,Alcôve,"Vinay, 51530",https://www.briqueteriechampagne.com/gastronomie,1 Star,1.0
1885,1885,Alexis Baudin,"Malling, 57480",https://alexisbaudin.fr,1 Star,1.0
1662,1662,Auffo,"Marseille, 13007",https://www.auffo-restaurant.com/,1 Star,1.0
1877,1877,Chalet Flachaire,"Abondance, 74360",https://www.chaletflachaire.fr,1 Star,1.0
1771,1771,Cheval Blanc,"Lembach, 67510",https://www.cheval-blanc-lembach.fr/,1 Star,1.0
1661,1661,Erre,"Chassy, 89110",https://roncemay.com,1 Star,1.0
1925,1925,Frédéric Molina à Forêt Ivre,"Vailly, 74470",https://foretivre.com,1 Star,1.0
1884,1884,Garrigue,"Ansouis, 84240",https://www.garrigue-restaurant.fr,1 Star,1.0


In [147]:
potential_removed_starred_entries[["source_row", "name", "location", "url", "award", "stars"]].sort_values(["stars", "name"], ascending=[False, True])


,source_row,name,location,url,award,stars
57,57,Bras,"Laguiole, 12210",http://www.bras.fr/fr/,2 Stars,2.0
97,97,Château de Beaulieu - Christophe Dufossé,"Busnes, 62350",https://www.lechateaudebeaulieu.fr,2 Stars,2.0
69,69,L'Abysse au Pavillon Ledoyen,"Paris, 75008",https://www.yannick-alleno.com/fr/,2 Stars,2.0
40,40,Palais Royal Restaurant,"Paris, 75001",https://palaisroyalrestaurantparis.com/,2 Stars,2.0
444,444,Auberge du Cheval Blanc,"Lembach, 67510",https://www.cheval-blanc-lembach.fr/fr/,1 Star,1.0
416,416,Domaine du Colombier,"Malataverne, 26780",https://www.domaine-colombier.com/,1 Star,1.0
455,455,Flaveurs,"Valence, 26000",http://www.baptistepoinot.com,1 Star,1.0
408,408,Frédéric Molina au Moulin de Léré,"Vailly, 74470",https://www.moulindelere.com/,1 Star,1.0
588,588,Guillaume Scheer - Les Plaisirs Gourmands,"Schiltigheim, 67300",http://www.les-plaisirs-gourmands.com,1 Star,1.0
224,224,Hélène Darroze à Villa La Coste,"Le Puy-Sainte-Réparade, 13610",https://www.villalacoste.com/,1 Star,1.0


## New high-star entries and material promotions/demotions

This keeps the historical focus on new two-star and three-star records, promotions, and demotions. It uses current-year rows for current restaurants and does not merge results back into the previous-year snapshot.


In [148]:
new_high_starred_entries = potential_new_starred_entries[potential_new_starred_entries["stars"].isin([2, 3])].copy()
new_high_starred_entries = new_high_starred_entries.sort_values(["stars", "name"], ascending=[False, True])

major_star_changes = pd.concat([
    star_changes.assign(name=star_changes["current_name"], location=star_changes["current_location"], stars=star_changes["current_stars"])[
        ["name", "location", "stars", "previous_stars", "status", "matching_method"]
    ],
    new_high_starred_entries.assign(previous_stars="New Entry", status="new_entry", matching_method="unmatched")[
        ["name", "location", "stars", "previous_stars", "status", "matching_method"]
    ],
], ignore_index=True)

major_star_changes.sort_values(["stars", "name"], ascending=[False, True], inplace=True)
major_star_changes


,name,location,stars,previous_stars,status,matching_method
0,Les Morainières,"Jongieux, 73170",3.00,2.0,promoted,name_postal
2,Alliance,"Paris, 75005",2.00,1.0,promoted,url_postal
3,Arbane,"Reims, 51100",2.00,1.0,promoted,name_postal
4,Bulle d'Osier,"Langres, 52200",2.00,1.0,promoted,name_location_url
5,Frédéric Doucet,"Charolles, 71120",2.00,1.0,promoted,name_postal
...,...,...,...,...,...,...
89,Restaurant Anne & Matthieu Omont - Hôtel de France,"Montmarault, 03390",0.25,0.5,demoted,name_postal
90,Sauf Imprévu,"Lyon, 69006",0.25,0.5,demoted,url_postal
91,Turbulent,"Trouville-sur-Mer, 14360",0.25,0.5,demoted,name_postal
92,Voyages des Sens,"Val-Revermont, 01370",0.25,0.5,demoted,url_postal


Historical note: for the 2025 France guide, Michelin's press announcement reported two new three-star restaurants and nine new two-star restaurants. That statement is useful context for manual verification, not executable proof that the notebook's heuristic matching is correct for every guide year.

Original manual observations such as “Blanc was a false fuzzy match” and “Le Puits Saint Jacques was demoted” are retained below as named review examples rather than dataframe row positions.


In [149]:
historical_manual_notes = pd.DataFrame([
    {
        "restaurant": "Blanc",
        "historical_observation": "False fuzzy-match in the original notebook; review by name/location instead of row index.",
    },
    {
        "restaurant": "Le Puits Saint Jacques",
        "historical_observation": "Original notebook manually added a demotion; verify using matched rows, name, and location rather than star_changes.loc[49].",
    },
])
historical_manual_notes


,restaurant,historical_observation
0,Blanc,False fuzzy-match in the original notebook; review by name/location instead of row index.
1,Le Puits Saint Jacques,"Original notebook manually added a demotion; verify using matched rows, name, and location rather than star_changes...."


In [150]:
def rows_matching_names(frame, names):
    wanted = [normalize_text(name) for name in names]
    mask = frame["normalized_name"].apply(lambda value: any(name in value or value in name for name in wanted))
    display_columns = ["guide_year", "source_row", "name", "location", "url", "award", "stars"]
    return frame.loc[mask, [column for column in display_columns if column in frame.columns]].sort_values(["name", "guide_year"])

manual_review_rows = pd.concat([
    rows_matching_names(previous_work, historical_manual_notes["restaurant"]),
    rows_matching_names(current_work, historical_manual_notes["restaurant"]),
], ignore_index=True)
manual_review_rows


,guide_year,source_row,name,location,url,award,stars
0,2025,2927,Amour Blanc,"Blois, 41000",https://fleurdeloire.com,Selected Restaurants,0.25
1,2025,2092,Au Cheval Blanc,"Hochstatt, 68720",https://www.au-cheval-blanc-hochstatt.com/,Selected Restaurants,0.25
2,2025,2780,Au Cheval Blanc,"Niedersteinbach, 67510",https://www.hotel-cheval-blanc.fr,Selected Restaurants,0.25
3,2025,444,Auberge du Cheval Blanc,"Lembach, 67510",https://www.cheval-blanc-lembach.fr/fr/,1 Star,1.00
4,2025,2303,Auberge du Cheval Blanc,"Bayonne, 64100",http://www.cheval-blanc-bayonne.com/,Selected Restaurants,0.25
5,2025,2718,Auberge du Cheval Blanc,"Westhalten, 68250",https://www.restaurant-koehler.com/,Selected Restaurants,0.25
6,2025,76,Blanc,"Paris, 75116",https://blanc-paris.com/,2 Stars,2.00
7,2025,2981,Cheval Blanc,"Feldbach, 68640",https://www.cheval-blanc-feldbach.fr/,Selected Restaurants,0.25
8,2025,226,Château Blanchard,"Chazelles-sur-Lyon, 42140",https://www.hotel-chateau-blanchard.com/,1 Star,1.00
9,2025,184,ES,"Paris, 75007",https://restaurant-es.com,1 Star,1.00


## Fuzzy review candidates

Fuzzy matching is useful for surfacing possible renamed restaurants, but it is not used here to silently consume additions or removals. Candidates are constrained by postal code, exact normalized address, or close coordinates when those fields are available. Several candidates may remain for one restaurant, and close scores should be reviewed manually.


In [151]:
def token_sort_ratio(left, right):
    left_tokens = " ".join(sorted(normalize_text(left).split()))
    right_tokens = " ".join(sorted(normalize_text(right).split()))
    return round(100 * SequenceMatcher(None, left_tokens, right_tokens).ratio())


def distance_km(left, right):
    required = ["latitude", "longitude"]
    if any(column not in left.index or column not in right.index for column in required):
        return None
    if pd.isna(left["latitude"]) or pd.isna(left["longitude"]) or pd.isna(right["latitude"]) or pd.isna(right["longitude"]):
        return None
    lat1, lon1 = math.radians(left["latitude"]), math.radians(left["longitude"])
    lat2, lon2 = math.radians(right["latitude"]), math.radians(right["longitude"])
    delta_lat, delta_lon = lat2 - lat1, lon2 - lon1
    value = math.sin(delta_lat / 2) ** 2 + math.cos(lat1) * math.cos(lat2) * math.sin(delta_lon / 2) ** 2
    return 6371.0 * 2 * math.atan2(math.sqrt(value), math.sqrt(1 - value))


FUZZY_THRESHOLD = 70
FUZZY_LIMIT_PER_CURRENT = 5

candidate_records = []
for _, current_row in potential_new_starred_entries.iterrows():
    scored = []
    for _, previous_row in previous_work[previous_work["source_row"].isin(previous_available)].iterrows():
        score = token_sort_ratio(current_row["name"], previous_row["name"])
        if score < FUZZY_THRESHOLD:
            continue
        same_postal = bool(current_row["postal_code"] and current_row["postal_code"] == previous_row["postal_code"])
        same_address = bool(current_row["normalized_address"] and current_row["normalized_address"] == previous_row["normalized_address"])
        kilometres = distance_km(previous_row, current_row)
        close_coordinates = kilometres is not None and kilometres <= 2
        if not (same_postal or same_address or close_coordinates):
            continue
        scored.append((score, kilometres, same_postal, same_address, previous_row))

    scored = sorted(scored, key=lambda item: (-item[0], item[1] if item[1] is not None else 999_999))[:FUZZY_LIMIT_PER_CURRENT]
    if not scored:
        continue
    best_score = scored[0][0]
    second_score = scored[1][0] if len(scored) > 1 else None
    for rank, (score, kilometres, same_postal, same_address, previous_row) in enumerate(scored, start=1):
        candidate_records.append({
            "current_name": current_row["name"],
            "current_location": current_row["location"],
            "current_stars": current_row["stars"],
            "previous_name": previous_row["name"],
            "previous_location": previous_row["location"],
            "previous_stars": previous_row["stars"],
            "name_score": score,
            "rank_for_current_record": rank,
            "score_gap_from_best": best_score - score,
            "gap_to_second_best": None if second_score is None else best_score - second_score,
            "same_postal_code": same_postal,
            "same_normalized_address": same_address,
            "distance_km": None if kilometres is None else round(kilometres, 3),
            "review_status": "needs_review",
        })

fuzzy_review_candidates = pd.DataFrame(candidate_records)
print(f"Fuzzy review candidates requiring manual review: {len(fuzzy_review_candidates)}")
fuzzy_review_candidates


Fuzzy review candidates requiring manual review: 5


,current_name,current_location,current_stars,previous_name,previous_location,previous_stars,name_score,rank_for_current_record,score_gap_from_best,gap_to_second_best,same_postal_code,same_normalized_address,distance_km,review_status
0,Monsieur Dior by Yannick Alléno,"Paris, 75008",1.0,Prunier par Yannick Alléno,"Paris, 75116",0.25,74,1,0,None,False,False,1.244,needs_review
1,Pantagruel,"Paris, 75001",1.0,Pantagruel,"Paris, 75002",1.00,100,1,0,None,False,False,1.000,needs_review
2,Pavyllon Paris,"Paris, 75008",1.0,Pavyllon,"Paris, 75008",1.00,73,1,0,None,True,True,0.000,needs_review
3,Frédéric Molina à Forêt Ivre,"Vailly, 74470",1.0,Frédéric Molina au Moulin de Léré,"Vailly, 74470",1.00,72,1,0,None,True,False,1.238,needs_review
4,Maison Fantin Latour - Stéphane Froidevaux,"Grenoble, 38000",1.0,Le Fantin Latour - Stéphane Froidevaux,"Grenoble, 38000",1.00,89,1,0,None,True,True,0.000,needs_review


In [152]:
if not fuzzy_review_candidates.empty:
    ambiguous_fuzzy_candidates = fuzzy_review_candidates[
        fuzzy_review_candidates["gap_to_second_best"].notna() & fuzzy_review_candidates["gap_to_second_best"].le(5)
    ].copy()
else:
    ambiguous_fuzzy_candidates = pd.DataFrame(columns=fuzzy_review_candidates.columns)

print(f"Ambiguous fuzzy candidates with a best-to-second-best gap <= 5: {len(ambiguous_fuzzy_candidates)}")
ambiguous_fuzzy_candidates


Ambiguous fuzzy candidates with a best-to-second-best gap <= 5: 0


,current_name,current_location,current_stars,previous_name,previous_location,previous_stars,name_score,rank_for_current_record,score_gap_from_best,gap_to_second_best,same_postal_code,same_normalized_address,distance_km,review_status


## Star-count comparison

This section intentionally compares the previous-year dataframe with the current-year dataframe. The old notebook accidentally derived both groups from the previous year.


In [153]:
previous_star_counts = france_previous[france_previous["stars"].ge(1)].groupby("stars").size()
current_star_counts = france_current[france_current["stars"].ge(1)].groupby("stars").size()

star_ratings = [1, 2, 3]
star_comparison_table = pd.DataFrame({
    "Star Rating": star_ratings,
    f"{PREVIOUS_YEAR} Guide": previous_star_counts.reindex(star_ratings, fill_value=0).astype(int).values,
    f"{YEAR} Guide": current_star_counts.reindex(star_ratings, fill_value=0).astype(int).values,
})
star_comparison_table


,Star Rating,2025 Guide,2026 Guide
0,1,538,547
1,2,78,81
2,3,30,30


## New restaurants by region

The visual helper is retained for continuity with the original notebook. It is fed with current-year rows for current restaurants, not with mutated previous-year data.


In [154]:
current_columns_for_visuals = [
    "name", "address", "location", "arrondissement", "department_num", "department", "capital", "region",
    "price", "cuisine", "url", "award", "stars", "longitude", "latitude",
]
current_columns_for_visuals = [column for column in current_columns_for_visuals if column in current_work.columns]

major_current_rows = current_work[current_work["name"].isin(major_star_changes["name"])][current_columns_for_visuals].copy()
major_current_rows = major_current_rows.merge(
    major_star_changes[["name", "location", "previous_stars", "status", "matching_method"]],
    on=["name", "location"], how="left",
)
major_current_rows


,name,address,location,arrondissement,department_num,department,capital,region,price,cuisine,url,award,stars,longitude,latitude,previous_stars,status,matching_method
0,Le P'tit Polyte,"Chalet Mounier, 2 rue de la Chapelle","Les Deux-Alpes, 38860",Grenoble,38,Isère,Grenoble,Auvergne-Rhône-Alpes,€€€€,Modern Cuisine,https://www.chalet-mounier.com/,Selected Restaurants,0.25,6.120844,45.003643,1.0,demoted,url_postal
1,La Table - Hôtel Clarance,32 rue de la Barre,"Lille, 59000",Lille,59,Nord,Lille,Hauts-de-France,€€€,Modern Cuisine,https://www.clarancehotel.com,Selected Restaurants,0.25,3.057182,50.638729,1.0,demoted,url_postal
2,Agapes,123 place de l'Église,"Chevagny-les-Chevrières, 71960",Mâcon,71,Saône-et-Loire,Mâcon,Bourgogne-Franche-Comté,€€,Modern Cuisine,https://agapes-atablerestaurant.com,Selected Restaurants,0.25,4.771371,46.332464,NaN,NaN,NaN
3,L’Écrin de Marlène,6 square de la Source-de-l'Hôpital,"Vichy, 03200",Vichy,03,Allier,Moulins,Auvergne-Rhône-Alpes,€€,Modern Cuisine,https://www.marlene-vichy.com,Selected Restaurants,0.25,3.421995,46.122797,0.5,demoted,name_postal
4,Racines,9 rue Corneille,"Toulon, 83000",Toulon,83,Var,Toulon,Provence-Alpes-Côte d'Azur,€€,Traditional Cuisine,https://www.racines-restaurant-toulon.com,Selected Restaurants,0.25,5.933568,43.124024,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
109,La Méditerranée,2 place de l'Odéon,"Paris, 75006",6th (Luxembourg),75,Paris,Paris,Île-de-France,€€€,Seafood,https://www.la-mediterranee.com/,Selected Restaurants,0.25,2.338656,48.850019,0.5,demoted,url_postal
110,Auberge de la Tour,2 place de la Fontaine,"Marcolès, 15220",Aurillac,15,Cantal,Aurillac,Auvergne-Rhône-Alpes,€€€€,Modern Cuisine,https://www.aubergedela-tour.com/,Selected Restaurants,0.25,2.351958,44.781562,1.0,demoted,name_location_url
111,Au 14 Février,2 rue du Portail,"Saint-Valentin, 36100",Issoudun,36,Indre,Châteauroux,Centre-Val de Loire,€€€€,Modern Cuisine,https://www.sv-au14fevrier.com/,Selected Restaurants,0.25,1.864990,46.951719,1.0,demoted,url_postal
112,Maison Devaux,70 rue du Commerce,"Rion-des-Landes, 40370",Dax,40,Landes,Mont-de-Marsan,Nouvelle-Aquitaine,€€,Modern Cuisine,https://maisondevaux.com,Selected Restaurants,0.25,-0.923337,43.934919,0.5,demoted,url_postal


In [155]:
from Functions.functions_visualisation import top_restaurants

In [156]:
top_restaurants(major_current_rows, granularity="region", star_rating=3, top_n=5)

Only 1 unique regions found.

⭐⭐⭐ restaurants in Auvergne-Rhône-Alpes


Region: Auvergne-Rhône-Alpes
1 ⭐⭐⭐ Restaurant




In [157]:
top_restaurants(major_current_rows, granularity="region", star_rating=2, top_n=5)

Only 4 unique regions found.

Top 4 regions with most ⭐⭐ restaurants:


Region: Île-de-France
6 ⭐⭐ Restaurants





Region: Grand Est
2 ⭐⭐ Restaurants





Region: Bourgogne-Franche-Comté
1 ⭐⭐ Restaurant





Region: Hauts-de-France
1 ⭐⭐ Restaurant




## Optional disposable outputs

Saved outputs include both compared years in the filename and metadata columns. They are analysis artifacts only; they do not replace accepted annual products or canonical reports.


In [158]:
save_csv(star_changes, "star_changes")
save_csv(potential_new_starred_entries[["source_row", "name", "location", "url", "award", "stars"]], "potential_new_starred_entries")
save_csv(potential_removed_starred_entries[["source_row", "name", "location", "url", "award", "stars"]], "potential_removed_starred_entries")
save_csv(fuzzy_review_candidates, "fuzzy_review_candidates")
save_csv(star_comparison_table, "star_count_comparison")


SAVE=False: not writing /Users/Ian/Documents/Study/Stage3/Programming/Github/Michelin_Rated_Restaurants/notebooks/outputs/2026/star_changes_2025_2026.csv
SAVE=False: not writing /Users/Ian/Documents/Study/Stage3/Programming/Github/Michelin_Rated_Restaurants/notebooks/outputs/2026/potential_new_starred_entries_2025_2026.csv
SAVE=False: not writing /Users/Ian/Documents/Study/Stage3/Programming/Github/Michelin_Rated_Restaurants/notebooks/outputs/2026/potential_removed_starred_entries_2025_2026.csv
SAVE=False: not writing /Users/Ian/Documents/Study/Stage3/Programming/Github/Michelin_Rated_Restaurants/notebooks/outputs/2026/fuzzy_review_candidates_2025_2026.csv
SAVE=False: not writing /Users/Ian/Documents/Study/Stage3/Programming/Github/Michelin_Rated_Restaurants/notebooks/outputs/2026/star_count_comparison_2025_2026.csv


## Postscript: useful manual search

This is retained from the original notebook as an exploratory check. Searches are run against both annual dataframes, and the displayed labels are dynamic.


In [159]:
def find_potential_matches(search_name, frame, threshold=70):
    rows = []
    for _, row in frame.iterrows():
        score = token_sort_ratio(search_name, row["name"])
        if score >= threshold:
            rows.append({
                "name": row["name"],
                "location": row["location"],
                "award": row.get("award"),
                "stars": row["stars"],
                "score": score,
            })
    return pd.DataFrame(rows).sort_values(["score", "name"], ascending=[False, True])

In [160]:
search_name = "Le Coquillage"
threshold = 70

matches_previous = find_potential_matches(search_name, france_previous, threshold)
matches_current = find_potential_matches(search_name, france_current, threshold)

print(f"Potential matches in {PREVIOUS_YEAR} data:")
display(matches_previous)

print(f"Potential matches in {YEAR} data:")
display(matches_current)

Potential matches in 2025 data:


,name,location,award,stars,score
0,Le Coquillage,"Saint-Méloir-des-Ondes, 35350",3 Stars,3.00,100
3,Le Collet,"Col de la Schlucht, 88400",Selected Restaurants,0.25,73
2,La Courtille,"Tavel, 30126",Selected Restaurants,0.25,72
1,Le Cottage,"Chonas-l'Amballan, 38121",Bib Gourmand,0.50,70


Potential matches in 2026 data:


,name,location,award,stars,score
2,Le Coquillage,"Saint-Méloir-des-Ondes, 35350",3 Stars,3.00,100
1,Le Collet,"Col de la Schlucht, 88400",Selected Restaurants,0.25,73
0,La Courtille,"Tavel, 30126",Selected Restaurants,0.25,72
3,Le Cottage,"Chonas-l'Amballan, 38121",Bib Gourmand,0.50,70


----
